In [0]:
USE databricks_ws.silver;

In [0]:
SELECT * FROM databricks_ws.silver.movie_genres;

movie_id,genre_id,file_date,ingestion_date,environment
17130,12,2024-12-16,2026-11-12T00:20:27.787205Z,Production
17130,18,2024-12-16,2026-11-12T00:20:27.787205Z,Production
17130,28,2024-12-16,2026-11-12T00:20:27.787205Z,Production
17130,35,2024-12-16,2026-11-12T00:20:27.787205Z,Production
17130,10402,2024-12-16,2026-11-12T00:20:27.787205Z,Production
17130,10749,2024-12-16,2026-11-12T00:20:27.787205Z,Production
17130,10751,2024-12-16,2026-11-12T00:20:27.787205Z,Production
15173,12,2024-12-16,2026-11-12T00:20:27.787205Z,Production
15173,14,2024-12-16,2026-11-12T00:20:27.787205Z,Production
15173,16,2024-12-16,2026-11-12T00:20:27.787205Z,Production


In [0]:
CREATE TABLE IF NOT EXISTS databricks_ws.gold.bridge_movie_genres
USING DELTA
AS
SELECT
    movie_id,
    genre_id
FROM databricks_ws.silver.movie_genres
WHERE 1 = 0;

num_affected_rows,num_inserted_rows


In [0]:

MERGE INTO databricks_ws.gold.bridge_movie_genres AS tgt
USING (
    SELECT DISTINCT
        mg.movie_id,
        mg.genre_id
    FROM databricks_ws.silver.movie_genres mg
    INNER JOIN databricks_ws.gold.dim_movies m
        ON mg.movie_id = m.movie_id
    INNER JOIN databricks_ws.gold.dim_genres g
        ON mg.genre_id = g.genre_id
    WHERE mg.file_date = :file_date
       OR EXISTS (
           SELECT 1
           FROM databricks_ws.silver.movies sm
           WHERE sm.movie_id = mg.movie_id
             AND sm.file_date = :file_date
       )
) AS src
ON tgt.movie_id = src.movie_id
   AND tgt.genre_id = src.genre_id

WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2102,532,0,1570


In [0]:
SELECT * FROM databricks_ws.gold.bridge_movie_genres;

movie_id,genre_id
11904,18
10991,16
13006,27
13768,18
9825,878
3989,10402
9023,10751
6477,35
13811,53
10105,36
